# 04 Modeling (BUSA 310 Project 1)

**Input:** `data/processed/philly_sf_sales_clean.csv`
**Group decisions (2026-10-12):**
- Drop all bundled sales (several parcels sold at one total price); keep vacant lots.
- Inputs: living area, bedrooms, bathrooms, ZIP, year built, interior and exterior condition, stories, garage spaces, central air, quality grade, sale year.
  `market_value` excluded (city's 2026 estimate, made after many of these sales: leakage).
- Target: compare price in dollars and log(price).
- Models: Linear Regression, Ridge, Random Forest, Gradient Boosting, plus a median baseline.
- Split: random 80/20, `random_state=42`.

**How the comparison stays fair:** every model sees the same training rows and is scored on the same test rows, in dollars.
Missing values are filled and categories encoded *inside* each model pipeline, so they are learned from training data only.
Model comparison uses 5-fold cross-validation on the training set; the test set is scored once at the end.

The numbers are printed; choosing the final model and interpreting the results is the group's job.

AI use note: modeling code drafted with Claude (Claude Code) from the group's stated choices; see AI Use Log.

## 0. Setup and load

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import sklearn
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

SEED = 42
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
CLEAN_PATH = "data/processed/philly_sf_sales_clean.csv"
if not os.path.exists(CLEAN_PATH):          # Colab starts empty: upload the cleaned file
    from google.colab import files
    os.makedirs("data/processed", exist_ok=True)
    print("Upload philly_sf_sales_clean.csv")
    name = list(files.upload().keys())[0]
    os.replace(name, CLEAN_PATH)

df = pd.read_csv(CLEAN_PATH, low_memory=False,
                 dtype={"zip_code": str, "parcel_number": str, "census_tract": str},
                 parse_dates=["sale_date"])
print("loaded:", df.shape)

pandas 3.0.5 | scikit-learn 1.9.1


loaded: (37469, 78)


## 1. Apply the group's modeling-scope decision: drop bundled sales

In [ ]:
before = len(df)
df = df[df["bundle_size"] == 1].copy()
print(f"bundled sales removed: {before - len(df):,}  ->  rows for modeling: {len(df):,}")
print("vacant lots kept:", int(df["flag_vacant"].sum()))

bundled sales removed: 7,897  ->  rows for modeling: 29,572
vacant lots kept: 981


## 2. Features and target
- Numeric inputs are used as numbers. Missing values are filled with the training median.
- ZIP, condition codes, central air and quality grade are treated as **categories** (one-hot encoded).
  Condition codes are categories because the code numbers are labels; check the OPA metadata before treating them as an ordered scale.
- Unknown categories at prediction time (for example a ZIP not in training) are ignored instead of crashing.
- Note: 0 bedrooms / 0 bathrooms are kept as recorded (open decision from cleaning).

In [ ]:
NUMERIC = ["total_livable_area", "number_of_bedrooms", "number_of_bathrooms", "year_built",
           "number_stories", "garage_spaces", "sale_year"]
CATEGORICAL = ["zip_code", "interior_condition", "exterior_condition", "central_air", "quality_grade"]
TARGET = "sale_price"

X = df[NUMERIC + CATEGORICAL].copy()
for c in ["interior_condition", "exterior_condition"]:
    X[c] = X[c].astype("Int64").astype(str).replace("<NA>", np.nan)   # codes as text labels
y = df[TARGET]
print(X.shape)
print("missing values per input:")
print(X.isna().sum().to_string())

(29572, 12)
missing values per input:
total_livable_area        3
number_of_bedrooms       67
number_of_bathrooms    1001
year_built                3
number_stories            8
garage_spaces           910
sale_year                 0
zip_code                  1
interior_condition        7
exterior_condition        5
central_air            5346
quality_grade          1580


## 3. Train/test split (random 80/20, done once, before any fitting)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"train: {len(X_train):,} rows | test: {len(X_test):,} rows")
print("same parcel in both sets:", len(set(df.loc[X_train.index, "parcel_number"]) & set(df.loc[X_test.index, "parcel_number"])))
print(f"median price  train ${y_train.median():,.0f} | test ${y_test.median():,.0f}")

train: 23,657 rows | test: 5,915 rows
same parcel in both sets: 0
median price  train $260,000 | test $255,001


## 4. Pipelines
`make_pipeline(model, log_target)` builds: fill missing values -> (scale numbers for linear models) -> encode categories -> model.
With `log_target=True`, the model learns log(price) and its predictions are converted back with exp() so every score is in dollars.
Note: exp() of a predicted log value estimates something closer to the *median* price for those features, so log models tend to predict slightly low on average. Section 7 checks this with mean signed error.

In [ ]:
def make_pipeline(model, log_target=False, scale=False):
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    prep = ColumnTransformer([
        ("num", Pipeline(num_steps), NUMERIC),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CATEGORICAL),
    ])
    pipe = Pipeline([("prep", prep), ("model", model)])
    if log_target:
        return TransformedTargetRegressor(regressor=pipe, func=np.log, inverse_func=np.exp)
    return pipe

MODELS = {
    "Linear Regression": lambda: (LinearRegression(), True),
    "Ridge (alpha=1)":   lambda: (Ridge(alpha=1.0), True),
    "Random Forest":     lambda: (RandomForestRegressor(n_estimators=100, min_samples_leaf=2,
                                                        n_jobs=-1, random_state=SEED), False),
    "Gradient Boosting": lambda: (HistGradientBoostingRegressor(random_state=SEED), False),
}
CANDIDATES = {}
for name, build in MODELS.items():
    for log_t in [False, True]:
        model, scale = build()
        CANDIDATES[f"{name} | {'log(price)' if log_t else 'price'}"] = make_pipeline(model, log_t, scale)
print(len(CANDIDATES), "model + target combinations")

8 model + target combinations


## 5. Baseline: predict every home at the training median
Any useful model must beat this.

In [ ]:
def scores(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred)}

baseline = DummyRegressor(strategy="median").fit(X_train, y_train)
base_scores = scores(y_test, baseline.predict(X_test))
print({k: (f"${v:,.0f}" if k != "R2" else round(v, 3)) for k, v in base_scores.items()})

{'MAE': '$186,424', 'RMSE': '$366,908', 'R2': -0.042}


## 6. Model comparison with 5-fold cross-validation (training data only)
Each model is trained 5 times on 4/5 of the training rows and scored on the remaining 1/5, all in dollars.
The test set is not used here.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
rows = []
for label, pipe in CANDIDATES.items():
    t = time.time()
    r = cross_validate(pipe, X_train, y_train, cv=cv, n_jobs=1,
                       scoring={"MAE": "neg_mean_absolute_error",
                                "RMSE": "neg_root_mean_squared_error", "R2": "r2"})
    rows.append({"model | target": label,
                 "CV MAE": -r["test_MAE"].mean(), "CV RMSE": -r["test_RMSE"].mean(),
                 "CV R2": r["test_R2"].mean(), "CV R2 std": r["test_R2"].std(),
                 "seconds": round(time.time() - t)})
    print(f"done: {label} ({rows[-1]['seconds']}s)")
cv_table = pd.DataFrame(rows).sort_values("CV RMSE")
cv_table.style.format({"CV MAE": "${:,.0f}", "CV RMSE": "${:,.0f}", "CV R2": "{:.3f}", "CV R2 std": "{:.3f}"})

done: Linear Regression | price (1s)


done: Linear Regression | log(price) (1s)


done: Ridge (alpha=1) | price (0s)


done: Ridge (alpha=1) | log(price) (0s)


done: Random Forest | price (25s)


done: Random Forest | log(price) (24s)


done: Gradient Boosting | price (4s)


done: Gradient Boosting | log(price) (5s)


,model | target,CV MAE,CV RMSE,CV R2,CV R2 std,seconds
6,Gradient Boosting | price,"$78,158","$140,297",0.826,0.025,4
4,Random Forest | price,"$77,048","$143,695",0.818,0.023,25
7,Gradient Boosting | log(price),"$77,562","$147,318",0.808,0.031,5
5,Random Forest | log(price),"$79,873","$156,541",0.783,0.030,24
0,Linear Regression | price,"$102,361","$190,051",0.682,0.043,1
2,Ridge (alpha=1) | price,"$101,043","$190,533",0.680,0.043,0
3,Ridge (alpha=1) | log(price),"$10,474,233","$714,188,014",-21268073.931,42536142.817,0
1,Linear Regression | log(price),"$12,437,402","$849,158,374",-30062907.217,60125802.543,1


## 7. Final scoring on the held-out test set
Every combination is refit on the full training set and scored once on the same test rows.
- **Mean signed error** = average of (predicted - actual). Positive = predicts too high on average; negative = too low.
- **Negative predictions** counts predicted prices below $0 (impossible values).

In [ ]:
test_rows, fitted = [], {}
for label, pipe in CANDIDATES.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[label] = pipe
    test_rows.append({"model | target": label, **{f"Test {k}": v for k, v in scores(y_test, pred).items()},
                      "Mean signed error": float(np.mean(pred - y_test)),
                      "Negative predictions": int((pred < 0).sum())})
test_rows.append({"model | target": "Baseline (training median)", **{f"Test {k}": v for k, v in base_scores.items()},
                  "Mean signed error": float(np.mean(baseline.predict(X_test) - y_test)), "Negative predictions": 0})
test_table = pd.DataFrame(test_rows).sort_values("Test RMSE")
os.makedirs("figures", exist_ok=True)
test_table.merge(cv_table, on="model | target", how="left").to_csv("figures/model_comparison.csv", index=False)
test_table.style.format({"Test MAE": "${:,.0f}", "Test RMSE": "${:,.0f}", "Test R2": "{:.3f}",
                         "Mean signed error": "${:,.0f}"})

,model | target,Test MAE,Test RMSE,Test R2,Mean signed error,Negative predictions
4,Random Forest | price,"$77,149","$167,630",0.782,$326,0
5,Random Forest | log(price),"$77,623","$170,905",0.774,"$-18,550",0
6,Gradient Boosting | price,"$80,091","$172,837",0.769,$986,0
7,Gradient Boosting | log(price),"$77,450","$175,054",0.763,"$-18,531",0
2,Ridge (alpha=1) | price,"$102,688","$211,217",0.655,$230,94
0,Linear Regression | price,"$104,423","$212,854",0.649,$747,112
8,Baseline (training median),"$186,424","$366,908",-0.042,"$-74,062",0
3,Ridge (alpha=1) | log(price),"$102,239","$401,036",-0.245,"$-7,669",0
1,Linear Regression | log(price),"$106,776","$510,702",-1.020,"$-1,925",0


## 8. Train vs test check (overfitting)
A large gap between training and test error means the model memorized the training data.

In [ ]:
gap = []
for label, pipe in fitted.items():
    tr = scores(y_train, pipe.predict(X_train)); te = scores(y_test, pipe.predict(X_test))
    gap.append({"model | target": label, "Train MAE": tr["MAE"], "Test MAE": te["MAE"],
                "Train R2": tr["R2"], "Test R2": te["R2"]})
pd.DataFrame(gap).style.format({"Train MAE": "${:,.0f}", "Test MAE": "${:,.0f}", "Train R2": "{:.3f}", "Test R2": "{:.3f}"})

,model | target,Train MAE,Test MAE,Train R2,Test R2
0,Linear Regression | price,"$101,299","$104,423",0.714,0.649
1,Linear Regression | log(price),"$94,415","$106,776",0.578,-1.020
2,Ridge (alpha=1) | price,"$100,173","$102,688",0.707,0.655
3,Ridge (alpha=1) | log(price),"$103,637","$102,239",-19.079,-0.245
4,Random Forest | price,"$40,431","$77,149",0.950,0.782
5,Random Forest | log(price),"$42,589","$77,623",0.928,0.774
6,Gradient Boosting | price,"$73,350","$80,091",0.871,0.769
7,Gradient Boosting | log(price),"$73,153","$77,450",0.838,0.763


## 9. Save the group's chosen model for the website
Set `FINAL_MODEL` to the exact label from the tables above (the group's decision), then run this cell.
It saves the whole pipeline (cleaning steps + model) so the website uses exactly what was tested.

In [ ]:
FINAL_MODEL = None   # e.g. "Gradient Boosting | log(price)"  <- group fills in

if FINAL_MODEL is None:
    print("No model chosen yet. Set FINAL_MODEL to one of:")
    print("\n".join(CANDIDATES))
else:
    import joblib
    os.makedirs("app", exist_ok=True)
    final = CANDIDATES[FINAL_MODEL]          # already fit on the training set in section 7
    joblib.dump({"model": final, "numeric": NUMERIC, "categorical": CATEGORICAL,
                 "trained_on": "philly_sf_sales_clean.csv, bundles dropped, random 80/20 train split"},
                "app/model.joblib")
    sample = X_test.iloc[[0]]
    print("saved app/model.joblib | check prediction for the website:", sample.to_dict("records")[0],
          f"-> ${final.predict(sample)[0]:,.0f}")

No model chosen yet. Set FINAL_MODEL to one of:
Linear Regression | price
Linear Regression | log(price)
Ridge (alpha=1) | price
Ridge (alpha=1) | log(price)
Random Forest | price
Random Forest | log(price)
Gradient Boosting | price
Gradient Boosting | log(price)


## Group notes (write your own)
- Which model the group chose and why:
- What the errors mean for a realtor using the tool:
